# Prep route and stop gdf
1. only subset for operators by region
2. do aggregation?
3. subset to special routes
   if this is skipped, after step 2, can go straight to regression?

* https://xyzservices.readthedocs.io/en/stable/introduction.html

In [9]:
import gcsfs
import geopandas as gpd
import google.auth
import pandas as pd

import world_cup_vars as wc_vars
import C4_event_helpers as C4
import E1_new_prep as E1

credentials, _ = google.auth.default()
GCS_FILE_PATH = wc_vars.GCS_FILE_PATH

In [15]:
operator_df = pd.read_parquet(
    f"{GCS_FILE_PATH}fct_daily_schedule_rt_route_direction_summary_{wc_vars.event_name}.parquet",
    filesystem = gcsfs.GCSFileSystem(),
    columns = ["feed_key", "schedule_name", "route_name", "direction_id", "route_id"]
).drop_duplicates()

route_df = pd.read_parquet(
    f"{GCS_FILE_PATH}route_summary_sofi.parquet",
    filesystem = gcsfs.GCSFileSystem(),
    columns = [
        "schedule_name", 
        "route_name", "direction_id", 
        "is_route_near", "is_special_route"]
).drop_duplicates()

In [16]:
metric_cols = [
    # "n_hours_in_service",
    "arrivals_per_hour_owl",
    "arrivals_per_hour_early_am",
    "arrivals_per_hour_am_peak",
    "arrivals_per_hour_midday",
    "arrivals_per_hour_pm_peak",
    "arrivals_per_hour_evening",
    "arrivals_owl",
    "arrivals_early_am",
    "arrivals_am_peak",
    "arrivals_midday",
    "arrivals_pm_peak",
    "arrivals_evening",
    "route_id_array",  # "route_type_array",
    # "wheelchair_boarding", "location_type"
]

stop_gdf = gpd.read_parquet(
    f"{GCS_FILE_PATH}fct_daily_scheduled_stops_{wc_vars.event_name}.parquet",
    storage_options={"token": credentials},
    columns=[
        "service_date",
        "feed_key",
        "stop_id",
        "stop_name",
        "daily_arrivals",
        "geometry",
    ]
    + metric_cols,
).merge(
    operator_df[["feed_key", "schedule_name"]].drop_duplicates(),
    how = "inner",
).pipe(C4.tag_event_days_and_times, wc_vars.sofi_match_times)

In [17]:
arrivals_by_event_df = C4.aggregate_by_event_type(
    stop_gdf,
    group_cols=["schedule_name", "stop_id", "stop_name", "event_day", "day_type"],
    metric_cols=["daily_arrivals"],
)

arrivals_wide = C4.make_wide(
    arrivals_by_event_df,
    index_cols=["schedule_name", "stop_id", "stop_name"],
    pivot_cols=["day_type", "event_day"],
    value_cols=["daily_arrivals"],
).pipe(C4.merge_in_stop_geom, stop_gdf)


In [20]:
route_df.dtypes, operator_df.dtypes

(schedule_name       object
 route_name          object
 direction_id         int64
 is_route_near         bool
 is_special_route      bool
 dtype: object,
 feed_key         object
 schedule_name    object
 route_name       object
 direction_id      int64
 route_id         object
 dtype: object)

In [24]:
# this scan be function, move all the specific route_name to route_id stuff together
near_routes = pd.merge(
    operator_df[["schedule_name", "route_name", "route_id", "direction_id"]].drop_duplicates(),
    route_df[route_df.is_route_near==True][["schedule_name", "route_name", "direction_id", "is_route_near"]],
    on = ["schedule_name", "route_name", "direction_id"],
    how = "inner"
)


special_routes = pd.merge(
    operator_df[["schedule_name", "route_name", "route_id", "direction_id"]].drop_duplicates(),
    route_df[route_df.is_special_route==True][["schedule_name", "route_name", "direction_id", "is_special_route"]],
    on = ["schedule_name", "route_name", "direction_id"],
    how = "inner"
)

In [28]:
special_route_ids = special_routes.route_id.unique().tolist()
near_route_ids = near_routes.route_id.unique().tolist()

In [30]:
arrivals_wide.explode("route_id_array")

,schedule_name,stop_id,stop_name,route_id_array,geometry,daily_arrivals_weekday_non_event,daily_arrivals_weekend_non_event,daily_arrivals_weekday_event,daily_arrivals_weekend_event,change_daily_arrivals_weekday,change_daily_arrivals_weekend
0,BART Schedule,A10-1,Lake Merritt,"12, 4, 6",POINT (-122.26529 37.79730),163.0,135.78,163.0,136.0,0.0,0.2
1,BART Schedule,A10-2,Lake Merritt,"11, 3, 5",POINT (-122.26523 37.79736),163.0,135.78,163.0,136.0,0.0,0.2
2,BART Schedule,A20-1,Fruitvale,"12, 4, 6",POINT (-122.22412 37.77482),163.0,135.78,163.0,136.0,0.0,0.2
3,BART Schedule,A20-2,Fruitvale,"11, 3, 5",POINT (-122.22407 37.77487),163.0,135.78,163.0,136.0,0.0,0.2
4,BART Schedule,A30-1,Coliseum,"12, 4, 6",POINT (-122.19676 37.75356),163.0,135.78,163.0,136.0,0.0,0.2
...,...,...,...,...,...,...,...,...,...,...,...
25096,Torrance Schedule,992,Imperial Hwy at Nash St,8,POINT (-118.38699 33.93081),29.0,15.50,29.0,13.0,0.0,-2.5
25097,Torrance Schedule,993,Imperial Hwy at Douglas St,8,POINT (-118.38281 33.93082),29.0,15.50,29.0,13.0,0.0,-2.5
25098,Torrance Schedule,996,Imperial Hwy at Douglas St,8,POINT (-118.38361 33.93114),28.0,15.50,28.0,13.0,0.0,-2.5
25099,Torrance Schedule,998,190TH ST AT FIRMONA AV,2,POINT (-118.35802 33.85820),13.0,13.00,13.0,0.0,0.0,-13.0


In [ ]:
sofi_gdf = E1.full_route_cleaning(wc_vars.event_name, "sofi")

In [ ]:
from dotenv import dotenv_values

CARTO_TILE_URL = dotenv_values("_env")["CARTO_TILE_URL"]

In [ ]:
sofi_gdf.explore(
    "is_near",
    tiles = CARTO_TILE_URL,
    attr="(C) OpenStreetMap contributors (C) CARTO"
)

In [ ]:
levi_gdf = E1.full_route_cleaning(wc_vars.event_name, "Levi Stadium")

In [ ]:
levi_gdf.plot("is_near")

In [ ]:
levi_gdf.is_near.value_counts()

In [ ]:
sofi_trips = C4.filter_to_special_routes(
    sofi_gdf,
    route_name_dict = wc_vars.special_socal_routes_dict
)